# Lattice Effects and FBS Grading on Electron Mixing Fractions

Adds discrete hyperedge paths and graded FBS broadcast to refine qDP/hDP fractions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Golden ratio and base gradients
phi = (1 + np.sqrt(5)) / 2
base_gradient = 1.0
gradients = {
    'eDP': (phi - 1) * base_gradient,
    'hDP': base_gradient,
    'qDP': phi * base_gradient
}

N_k = 1.0
T = N_k
n_samples = 200000  # reduced for speed; increase later

# Lattice noise + FBS grading factor (r^{-2} falloff)
r_norm = np.random.uniform(0.1, 2.0, n_samples)  # normalized distance
fbs_grade = 1 / (r_norm**2 + 0.1)  # avoid div by zero
fbs_grade /= fbs_grade.max()  # normalize to [0,1]

noise_scale = 0.05 * fbs_grade  # grading modulates noise
noise = np.random.normal(0, noise_scale[:, np.newaxis], (n_samples, 3))

energies = np.array([gradients['eDP'], gradients['hDP'], gradients['qDP']]) + noise

# Boltzmann probabilities
probs = np.exp(-energies / T)
probs /= probs.sum(axis=1)[:, np.newaxis]

# Average fractions
avg_fracs = probs.mean(axis=0)
std_fracs = probs.std(axis=0)
labels = ['eDP', 'hDP', 'qDP']

print("Lattice-refined average fractions (electron, N_k=1):")
for label, avg, std in zip(labels, avg_fracs, std_fracs):
    print(f"{label}: {avg:.5f} ± {std:.5f}")

# Plot qDP distribution
plt.hist(probs[:, 2], bins=100, density=True, alpha=0.7, color='orange', label='qDP')
plt.axvline(avg_fracs[2], color='red', linestyle='--', label=f'Mean qDP = {avg_fracs[2]:.5f}')
plt.xlabel('qDP Fraction')
plt.ylabel('Density')
plt.title('Lattice Effects on qDP Distribution (N_k=1)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../figures/electron-qdp-lattice-effects.png')
plt.show()

## Preliminary g-2 Estimate with Lattice Effects
Using refined fractions

In [ ]:
C = 9.6e-7
alpha = 1 / 137.035999084
S = alpha / (2 * np.pi)

f_qDP = avg_fracs[2]
f_hDP = avg_fracs[1]
mixing_sum = f_qDP + 0.7 * f_hDP
raw_delta = C * mixing_sum
delta_mu_e = raw_delta * S

print(f"Raw mixing boost: {raw_delta:.2e}")
print(f"Suppression S: {S:.4e}")
print(f"Electron δμ with lattice effects: {delta_mu_e:.2e}")